## Transform US Census GeoSpatial data on Hugging Face to parquet using DuckDB

Here we'll do the following:
- download the zipped shapefile from Hugging Face
- transfrom the zipped shapefile to parquet using DuckDB
- query the parquet files using DuckDB
- push the parquet files to Hugging Face
- query the remote parquet files using DuckDB



In [1]:
import duckdb
import os
from google.colab import userdata


In [ ]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [3]:
roads_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.zip"
states_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.zip"


In [4]:
# Extract the base file name
roads_base = roads_url.split("/")[-1].split(".")[0]
states_base = states_url.split("/")[-1].split(".")[0]

roads_base, states_base


('tl_2024_35006_roads', 'tl_2024_us_state')

In [5]:
! curl -L -O {roads_url}
! curl -L -O {states_url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1002  100  1002    0     0   3445      0 --:--:-- --:--:-- --:--:--  3455
100 2368k  100 2368k    0     0  2683k      0 --:--:-- --:--:-- --:--:-- 6042k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   998  100   998    0     0   7372      0 --:--:-- --:--:-- --:--:--  7392
100 9721k  100 9721k    0     0  11.3M      0 --:--:-- --:--:-- --:--:-- 57.0M


In [6]:
! ls -la {roads_base}* {states_base}*


-rw-r--r-- 1 root root 2425481 Nov 19 04:49 tl_2024_35006_roads.zip
-rw-r--r-- 1 root root 9954307 Nov 19 04:49 tl_2024_us_state.zip


In [7]:
roads_path = f"/vsizip/{roads_base}.zip"
states_path = f"/vsizip/{states_base}.zip"

roads_path, states_path


('/vsizip/tl_2024_35006_roads.zip', '/vsizip/tl_2024_us_state.zip')

## Convert to Parquet


In [8]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

! ls -la {roads_base}.parquet

-rw-r--r-- 1 root root 2317411 Nov 19 04:49 tl_2024_35006_roads.parquet


In [9]:
# Export States shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read("{states_path}")
    ) TO '{states_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

! ls -la {states_base}.parquet

-rw-r--r-- 1 root root 11340375 Nov 19 04:49 tl_2024_us_state.parquet


In [10]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_base}.parquet') roads
    LEFT JOIN read_parquet('{states_base}.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'
"""
print(query)

gdf_combined = con.sql(query).df()
gdf_combined



    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('tl_2024_35006_roads.parquet') roads
    LEFT JOIN read_parquet('tl_2024_us_state.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'



,LINEARID,FULLNAME,RTTYP,MTFCC,STATEFP,STUSPS,state_name,road_geometry,state_geometry
0,1104492838447,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 52, 251, 215, 194, 68...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."
1,1104492838321,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 4, 251, 215, 194, 129...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [11]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [12]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


'DUC-2026'

In [13]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


'hf_ ... dQPb'

In [14]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push the parquet files to Hugging Face


### State

In [15]:
hf_file = os.environ["HF_FILE"] = f"{states_base}.parquet"
hf_file


'tl_2024_us_state.parquet'

In [16]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [17]:
print(hf_upload.stdout)


DRY-RUN: hf upload --type dataset rwcitek/DUC-2026 ./tl_2024_us_state.parquet ./tl_2024_us_state.parquet



### Roads

In [18]:
hf_file = os.environ["HF_FILE"] = f"{roads_base}.parquet"
hf_file


'tl_2024_35006_roads.parquet'

In [19]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [20]:
print(hf_upload.stdout)


DRY-RUN: hf upload --type dataset rwcitek/DUC-2026 ./tl_2024_35006_roads.parquet ./tl_2024_35006_roads.parquet



## Verify


In [21]:
roads_parquet_url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.parquet'
states_parquet_url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.parquet'

roads_parquet_url, states_parquet_url


('https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.parquet',
 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.parquet')

In [22]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_parquet_url}') roads
    LEFT JOIN read_parquet('{states_parquet_url}') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'
"""
print(query)

gdf_combined = con.sql(query).df()
gdf_combined



    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.parquet') roads
    LEFT JOIN read_parquet('https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'



,LINEARID,FULLNAME,RTTYP,MTFCC,STATEFP,STUSPS,state_name,road_geometry,state_geometry
0,1104492838447,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 52, 251, 215, 194, 68...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."
1,1104492838321,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 4, 251, 215, 194, 129...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."


## Describe parquet files


In [23]:
duckdb.query(f'''
  describe
  select *
  from read_parquet("{roads_parquet_url}")
  ;
''')


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ LINEARID    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FULLNAME    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ RTTYP       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ MTFCC       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geom        │ BLOB        │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [24]:
duckdb.query(f'''
  describe
  select *
  from read_parquet("{states_parquet_url}")
  ;
''')


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ REGION      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ DIVISION    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ STATEFP     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ STATENS     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ GEOID       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ GEOIDFQ     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ STUSPS      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ NAME        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LSAD        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ MTFCC       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FUNCSTAT    │ VARC